[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github&logoColor=white)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_Colab.ipynb)


# 🎬 MiniMax-H3 — Video + Audio Generation (33B, INT8 quantized)

A Colab port of [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) — a 33B parameter video generation model that produces **video with synchronized audio** (ambience, foley, speech). Supports text-to-video, image-to-video (first/last frame), and reference-based generation.

## How it works

MiniMax-H3 uses a **split deployment** architecture:

1. **Conditioner** — encodes the text prompt + optional keyframe images into `prompt_embeds` + `text_token_tags`. Two modes:
   - **Remote** (default): calls the `multimodalart/qwen3vl-conditioner` HF Space via `gradio_client`. No local text encoder needed (saves 62 GB download + VRAM). Depends on HuggingFace ZeroGPU quota.
   - **Local**: loads the 62 GB Qwen3-VL text encoder locally with INT8 quantization (~31 GB), runs the conditioning step, then frees the text encoder before loading the transformer. No ZeroGPU dependency. Requires downloading an additional 62 GB of weights (cached on Drive).

2. **Local denoiser** (on your Colab GPU): the 33B transformer (DiT) denoises the video+audio latents, then the VAEs decode them into frames + stereo audio. Uses the official diffusers `minimax-h3` branch with **INT8 quantization + block-level group offloading** (from the [official docs](https://github.com/huggingface/diffusers/blob/minimax-h3/docs/source/en/api/pipelines/minimax_h3.md)).

```
prompt + images → [remote conditioner HF Space] → prompt_embeds
                                                    ↓
              INT8 DiT (group_offload) → denoise → VAEs → video.mp4 + audio
```

## ⚠️ License — Territory Restriction + MAU Cap

MiniMax H3 Community License:
- **Excludes:** EU, UK, South Korea, **and USA**
- **>1M MAU** requires separate commercial license
- Continuing past the header cell is your acceptance of the license

## Quick start

1. **Runtime → Change runtime type → GPU** (L4, A100, or A100 80GB)
2. Run **STEP 1** — installs torch 2.11.0+cu128, diffusers from `minimax-h3` branch, torchao for INT8 quantization. First run: ~10-15 min.
3. Run **STEP 2** — downloads FL2VA transformer (61.7 GB) + VAEs (10.3 GB) to Drive cache. First run: ~30-60 min.
4. Run **STEP 3** — imports, remote conditioner, lazy model loader with INT8 + group offload
5. Run **STEP 4** — opens the Gradio UI. Enter a prompt, pick canvas/duration/steps, click Generate.
6. **STEP 5** keep-alive, **STEP 6** quick test, **STEP 7** batch

## Memory

The official docs provide recipes for different GPU sizes:

| GPU | VRAM | Recipe | Peak VRAM |
|-----|------|--------|-----------|
| **A100 80GB** | 80 GB | `auto_cpu_offload` (no quantization) | ~68 GB |
| **A100 40GB** | 40 GB | INT8 + `group_offload(block_level)` | ~18 GB |
| **L4 22GB** | 22 GB | INT8 + `group_offload(block_level)` + small canvas | ~15 GB |

The notebook auto-detects GPU VRAM and picks the right recipe. Smaller canvases (960×544) run ~2.3x faster per step than the trained 1344×768.

## Outputs

```
output.mp4    # Video + synchronized stereo audio (32 kHz)
```

## Technical notes

- **diffusers from `minimax-h3` branch**: MiniMax-H3 support is in an open PR (#14355), not on PyPI. We install from the branch directly.
- **INT8 quantization**: Uses `TorchAoConfig(Int8WeightOnlyConfig(version=2))` from the official docs. The `version=2` tensors are pinnable, which streamed offload needs.
- **Block-level group offloading**: `enable_group_offload(offload_type="block_level", num_blocks_per_group=1)` moves one transformer block at a time to GPU. This is the key to fitting on L4/A100-40GB.
- **Remote conditioner**: The 62 GB Qwen3-VL text encoder runs on the HF Space. We call it via `gradio_client`. This saves 62 GB download + 62 GB VRAM.
- **`spaces` stub**: The upstream code uses `import spaces` for HF ZeroGPU. We install a stub module so the code runs on Colab.
- **PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True**: Reduces memory fragmentation.

## Companion notebooks

- **Wan2.2_Colab** — text/image-to-video (no audio)
- **Wan2.2_Animate_Colab** — character animation
- **GaussianGPT_Colab** — autoregressive 3D scene generation
- **InfiniSplat_Colab** — single-image 3DGS reconstruction


In [ ]:
#@title STEP 1 — Install torch 2.11.0+cu128, diffusers PR #14355 head, torchao, transformers
"""
• Pins torch to 2.11.0+cu128 (the MiniMax-H3 release needs a recent torch)
• Installs diffusers from PR #14355 head (MiniMax-H3 is now in MODULAR_PIPELINE_MAPPING,
  so the package-level `MiniMaxH3ModularPipeline` export resolves correctly)
• Installs torchao for INT8 quantization (Int8WeightOnlyConfig)
• Pins transformers 5.8.0 (required for Qwen3-VL processor)
• Installs PyAV for video+audio muxing
• Stubs the `spaces` module (HF ZeroGPU API, not available on Colab)
• Mounts Google Drive for checkpoint caching
"""
import os, sys, time, subprocess, pathlib, types

print('='*72)
print('MiniMax-H3 — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MiniMax-H3')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_h3_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')

OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-H3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT = pathlib.Path('/content/h3_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f'  Recipe        : ComponentsManager + auto offload (A100 80GB) or INT8 + group_offload (24-40GB)')

t_total = time.time()

# 1. Pin torch to 2.11.0+cu128 ───────────────────────────────────────────
TARGET_TORCH = '2.11.0'
if not torch.__version__.startswith(TARGET_TORCH):
    print(f'\n[1/5] Pinning torch to {TARGET_TORCH}+cu128 ...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-input',
        f'torch=={TARGET_TORCH}+cu128',
        'torchvision==0.26.0+cu128',
        'torchaudio==2.11.0+cu128',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--force-reinstall',
    ], check=False)
    print(f'  torch installed in {time.time()-t0:.1f}s')
    print('  Restarting kernel to load new torch ...')
    import os as _os
    _os.execv(sys.executable, [sys.executable] + sys.argv)
else:
    print(f'\n[1/5] torch {torch.__version__} already matches — fixing huggingface_hub')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'huggingface-hub>=1.25.0',
    ], check=False)
    print(f'  huggingface-hub>=1.25.0 installed in {time.time()-t0:.1f}s')

# 2. Install diffusers from PR #14355 head (merged in commit f53d552). ──
# MiniMax-H3 is now integrated into the official huggingface/diffusers
# MODULAR_PIPELINE_MAPPING (model_name='minimax-h3' -> MiniMaxH3ModularPipeline),
# so the package-level import `from diffusers import ModularPipeline,
# MiniMaxH3Transformer3DModel, MiniMaxH3Scheduler, TorchAoConfig` works.
print('\n[2/5] Installing diffusers from PR #14355 head (MiniMax-H3 release) ...')
t0 = time.time()
# Use --no-deps to prevent diffusers from pulling conflicting deps.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'git+https://github.com/huggingface/diffusers.git@refs/pull/14355/head',
], check=False)
print(f'  diffusers installed in {time.time()-t0:.1f}s')

# 3. Install torchao + transformers + accelerate + other deps ───────────
# Force-reinstall torchao to rebuild against the new torch (after os.execv restart).
print('\n[3/5] Installing torchao + transformers + accelerate ...')
t0 = time.time()
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchao>=0.15.0', '--force-reinstall', '--no-deps',
], check=False)
EXTRA_PKGS = [
    'transformers==5.8.0',
    'accelerate==1.14.0',
    'huggingface-hub>=1.25.0',
    'safetensors>=0.8.0',
    'av',
    'einops',
    'omegaconf',
    'gradio>=5.49.1,<7',
    'pillow',
    'scipy',
    'opencv-python-headless',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  Extra deps installed in {time.time()-t0:.1f}s')

# 4. Stub the `spaces` module (HF ZeroGPU, not available on Colab) ─────
print('\n[4/5] Stubbing `spaces` module ...')
spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = spaces_stub
print('  spaces stub installed')

# 5. Verify imports ─────────────────────────────────────────────────────
print('\n[5/5] Verifying imports ...')
t0 = time.time()
try:
    import diffusers
    print(f'  diffusers    : {diffusers.__version__}')
except ImportError as e:
    print(f'  [FAIL] diffusers: {e}')
try:
    import torchao
    print(f'  torchao      : {torchao.__version__}')
except ImportError as e:
    print(f'  [WARN] torchao: {e}')
try:
    import transformers
    print(f'  transformers  : {transformers.__version__}')
except ImportError as e:
    print(f'  [FAIL] transformers: {e}')
try:
    import av
    print(f'  av           : OK')
except ImportError as e:
    print(f'  [FAIL] av: {e}')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache  : {drive_root}')
print(f'  Output dir   : {OUT_DIR}')
print()
print('Next: run STEP 2 (download FL2VA weights).')


In [ ]:
#@title STEP 2 — Download FL2VA transformer + VAEs to Drive cache
"""
Downloads the MiniMax-H3 weights to Drive cache.

Downloads the full MiniMax-H3 checkpoint (~134 GB on disk): both transformer
partitions, both VAEs, the Qwen3-VL text encoder, the schedulers and the
"index plumbing". The pipeline will fetch components from this local cache.
"""
import os, sys, time, pathlib
from huggingface_hub import snapshot_download

print('='*72)
print('MiniMax-H3 — Download weights')
print('='*72)

MODEL_REPO = 'MiniMaxAI/MiniMax-H3'
CKPT_DIR = drive_root / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'  Model repo      : {MODEL_REPO}')
print(f'  Cache dir       : {CKPT_DIR}')
print()

# Full checkpoint set: transformer + transformer_ref + both VAEs + text_encoder +
# tokenizer / processor / chat template / schedulers. Lets ModularPipeline.from_pretrained
# resolve any of the three workflows (t2va, fl2va, ref2va) from local cache.
ALLOW_PATTERNS = [
    'modular_model_index.json',
    # Transformer (14 shards, ~61.7 GB)
    'transformer/*.safetensors',
    'transformer/config.json',
    'transformer/*.index.json',
    # Reference transformer (separate partition)
    'transformer_ref/*.safetensors',
    'transformer_ref/config.json',
    'transformer_ref/*.index.json',
    # Video VAE (3 shards, ~9.7 GB)
    'vae/*.safetensors',
    'vae/config.json',
    'vae/*.index.json',
    # Audio VAE (1 file, ~577 MB)
    'audio_vae/*.safetensors',
    'audio_vae/config.json',
    # Qwen3-VL text encoder (~62 GB bf16 — streamed off CPU on smaller cards)
    'text_encoder/*.safetensors',
    'text_encoder/config.json',
    'text_encoder/*.index.json',
    'text_encoder/chat_template.json',
    'text_encoder/preprocessor_config.json',
    'text_encoder/video_preprocessor_config.json',
    # Tokenizer / processor (tiny JSON)
    'tokenizer/*.json',
    'tokenizer/*.txt',
    'processor/*.json',
    'processor/*.txt',
    # Schedulers (tiny JSON files)
    'scheduler/scheduler_config.json',
    'audio_scheduler/scheduler_config.json',
]

t_total = time.time()
print(f'  Downloading {len(ALLOW_PATTERNS)} pattern groups ...')
print(f'  (This downloads ~134 GB. First run: 30-60 min depending on network.)')
print()

snapshot_download(
    repo_id=MODEL_REPO,
    allow_patterns=ALLOW_PATTERNS,
    local_dir=str(CKPT_DIR),
    cache_dir=os.environ.get('HF_HOME'),
    max_workers=4,
)

elapsed = time.time() - t_total

# Verify downloads
print()
print('  Downloaded files:')
total_size = 0
for f in sorted(CKPT_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total_size += sz
        rel = str(f.relative_to(CKPT_DIR))
        if sz > 1024**3:
            print(f'    {rel:>55s}  {sz/1024**3:.2f} GB')
        elif sz > 1024**2:
            print(f'    {rel:>55s}  {sz/1024**2:.1f} MB')

print()
print('='*72)
print(f'STEP 2 complete in {elapsed/60:.1f} min')
print(f'  Total downloaded: {total_size/1024**3:.1f} GB')
print(f'  Cache dir: {CKPT_DIR}')
print('='*72)
print()
print('Next: run STEP 3 (imports + lazy model loader + INT8 quantization).')


In [ ]:
#@title STEP 3 — Imports, spaces stub, lazy model loader + generate_video
"""
• Stubs the `spaces` module (HF ZeroGPU, not available on Colab)
• Defines `load_minimax_h3()` — loads the pipeline using the official diffusers
  PR #14355 release recipe:
    - A100 80GB: ComponentsManager + auto_cpu_offload (no quantization)
    - A100 40GB / L4: INT8 transformer + INT8 text_encoder on load,
      then transformer block-level group offload + text_encoder leaf-level
      group offload + VAEs resident on GPU
• Defines `generate_video()` — the full pipeline call: condition → denoise →
  decode → mux video+audio
• Auto-detects GPU VRAM and picks the right memory recipe
"""
import os, sys, time, gc, pathlib, types, traceback
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch

print('='*72)
print('MiniMax-H3 — Imports + lazy model loader')
print('='*72)

# --- Stub the `spaces` module (must come before any diffusers import) ──
_spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
_spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = _spaces_stub

# --- Verify deps ────────────────────────────────────────────────────────
import diffusers
import transformers
try:
    import torchao
    print(f'  diffusers    : {diffusers.__version__}')
    print(f'  transformers  : {transformers.__version__}')
    print(f'  torchao      : {torchao.__version__}')
except ImportError as e:
    print(f'  [FAIL] {e}')
    raise
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_VRAM_GB = p.total_memory / (1024**3)
    print(f'  GPU          : {p.name}  ({GPU_VRAM_GB:.1f} GB)')
else:
    GPU_VRAM_GB = 0
    print('  WARNING: no GPU detected')
print()

CKPT_DIR = drive_root / 'checkpoints'
MODEL_REPO = 'MiniMaxAI/MiniMax-H3'

    # --- Canvas definitions (resolution + aspect-ratio presets) ──────────
CANVASES = {
    "960x544 · 16:9 fast": (544, 960),
    "1024x576 · 16:9 fast": (576, 1024),
    "1152x640 · 16:9": (640, 1152),
    "1280x704 · 16:9": (704, 1280),
    "1344x768 · 16:9 full": (768, 1344),
    "544x960 · 9:16 fast": (960, 544),
    "640x1152 · 9:16": (1152, 640),
    "768x1344 · 9:16 full": (1344, 768),
    "544x544 · 1:1 fast": (544, 544),
    "768x768 · 1:1 full": (768, 768),
    "768x576 · 4:3 fast": (576, 768),
    "1024x768 · 4:3 full": (768, 1024),
    "576x768 · 3:4 fast": (768, 576),
    "768x1024 · 3:4 full": (1024, 768),
    "1152x512 · 21:9 fast": (512, 1152),
    "1536x672 · 21:9 full": (672, 1536),
}
DEFAULT_CANVAS = "960x544 · 16:9 fast"
FPS, FRAMES_PER_CHUNK, LATENTS_PER_CHUNK = 24, 17, 5
MAX_UI_DURATION = 14

def snap_frames(seconds):
    """The frame count MiniMax-H3's video VAE can decode: the next 17*n + 5 at 24 fps."""
    frames = max(1, round(float(seconds) * FPS))
    while frames % FRAMES_PER_CHUNK != LATENTS_PER_CHUNK:
        frames += 1
    return frames

# --- Conditioner dispatch (placeholder; the pipeline's text_encoder step does it now) ──
def call_conditioner(*args, **kwargs):
    """No-op stub: the official MiniMax-H3 pipeline runs Qwen3-VL internally.

    Kept so older UI code paths don't break. Returns the prompt unchanged.
    """
    return None, None, {'height': '0', 'width': '0', 'num_frames': '0', 'prompt': ''}, {}

# --- Lazy model loader (singleton) ────────────────────────────────────
_PIPE = None
_DEVICE = None

def load_minimax_h3(verbose=True):
    """Load the MiniMax-H3 pipeline.

    Uses the official diffusers release recipe (PR #14355):
    - A100 80GB: ComponentsManager + auto_cpu_offload (no quantization)
    - A100 40GB / L4: INT8 (quantize transformer + text_encoder on load,
      then transformer block-level group offload + text_encoder leaf-level
      group offload + VAEs resident on GPU)

    Returns: (pipe, device)
    """
    global _PIPE, _DEVICE
    if _PIPE is not None:
        return _PIPE, _DEVICE

    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    from diffusers import ModularPipeline, MiniMaxH3Transformer3DModel, TorchAoConfig
    from diffusers.modular_pipelines.components_manager import ComponentsManager
    from transformers import Qwen3VLForConditionalGeneration
    from transformers import TorchAoConfig as TransformersTorchAoConfig
    from torchao.quantization import Int8WeightOnlyConfig

    manager = ComponentsManager()

    if verbose:
        print(f'  [gen] Loading pipeline from {MODEL_REPO} (workflow=fl2va) ...')
    t0 = time.time()

    if GPU_VRAM_GB >= 70:
        # A100 80GB recipe: load everything in bf16, ComponentsManager evicts to CPU between components.
        if verbose:
            print(f'  [gen] bf16 + auto_cpu_offload recipe (GPU {GPU_VRAM_GB:.0f} GB) ...')
        pipe = ModularPipeline.from_pretrained(MODEL_REPO, workflow='fl2va', components_manager=manager)
        pipe.load_components(workflow='fl2va', dtype=torch.bfloat16)
        try:
            pipe.transformer.set_attention_backend('_flash_3_hub')  # Hopper ~3x faster
        except Exception:
            pipe.transformer.set_attention_backend('_native_cudnn')
        manager.enable_auto_cpu_offload(device=str(_DEVICE), memory_reserve_margin='12GB')
    else:
        # A100 40GB / L4 recipe: INT8 quantize transformer + text_encoder on the way in,
        # then stream the transformer blocks from CPU RAM one at a time.
        if verbose:
            print(f'  [gen] INT8 + group_offload recipe (GPU {GPU_VRAM_GB:.0f} GB) ...')

        # Instantiate the pipeline WITHOUT components yet, then inject quantized components.
        # NOTE: when quantizing, transformers auto-sets `low_cpu_mem_usage=True` and rejects any
        # explicit False/None — do NOT pass `low_cpu_mem_usage=...` here.
        pipe = ModularPipeline.from_pretrained(MODEL_REPO, components_manager=manager)
        pipe.update_components(
            transformer=MiniMaxH3Transformer3DModel.from_pretrained(
                MODEL_REPO, subfolder='transformer', dtype=torch.bfloat16,
                quantization_config=TorchAoConfig(
                    Int8WeightOnlyConfig(),
                    modules_to_not_convert=[
                        'proj_in', 'audio_proj_in', 'context_embedder', 'time_embedder',
                        'time_proj', 'token_refiner', 'norm_out', 'proj_out', 'audio_proj_out',
                    ],
                ),
                cache_dir=os.environ.get('HF_HOME'),
            ),
            text_encoder=Qwen3VLForConditionalGeneration.from_pretrained(
                MODEL_REPO, subfolder='text_encoder', dtype=torch.bfloat16,
                quantization_config=TransformersTorchAoConfig(
                    Int8WeightOnlyConfig(),
                    modules_to_not_convert=[
                        'model.visual', 'model.language_model.embed_tokens',
                        'model.language_model.norm', 'lm_head',
                    ],
                ),
                cache_dir=os.environ.get('HF_HOME'),
            ),
        )
        pipe.load_components(workflow='fl2va', dtype=torch.bfloat16)

        # version=2 int8 tensors are pinnable; freezing removes the autograd path the quantized tensors cannot serve.
        pipe.transformer.requires_grad_(False)
        pipe.text_encoder.requires_grad_(False)

        try:
            pipe.transformer.set_attention_backend('_native_cudnn')
        except Exception:
            pass

        offload = dict(onload_device=_DEVICE, offload_device=torch.device('cpu'), use_stream=False)
        pipe.transformer.enable_group_offload(offload_type='block_level', num_blocks_per_group=1, **offload)
        from diffusers.hooks import apply_group_offloading
        apply_group_offloading(pipe.text_encoder.model, offload_type='leaf_level', **offload)
        # VAEs are small (~10 GB combined) and stream-friendly — keep them on GPU.
        pipe.vae.to(_DEVICE)
        pipe.audio_vae.to(_DEVICE)

    if verbose:
        print(f'  [gen] Model loaded in {time.time()-t0:.1f}s')
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            print(f'  [gen] VRAM after setup: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

    _PIPE = pipe
    return _PIPE, _DEVICE

def free_minimax_h3():
    """Unload the model and free GPU memory."""
    global _PIPE, _DEVICE
    if _PIPE is not None:
        del _PIPE
    _PIPE = None
    _DEVICE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Full generation pipeline ─────────────────────────────────────────
def generate_video(prompt, image_path=None, last_image_path=None,
                     canvas=DEFAULT_CANVAS, duration=5, steps=28, seed=42,
                     rewrite_prompt=False, verbose=True):
    """Full MiniMax-H3 pipeline: condition (Qwen3-VL) → denoise → decode → mux.

    Pass `prompt` (a single string). The pipeline's text-encoder block runs
    the Qwen3-VL conditioner internally (or streams it from CPU on L4).

    Returns: (path_to_mp4, report_dict)
    """
    from PIL import Image, ImageOps
    from diffusers.utils import encode_video

    num_frames = snap_frames(duration)
    height, width = CANVASES[canvas]

    pipe, device = load_minimax_h3(verbose=verbose)

    # Keyframes are EXIF-transposed and converted to RGB before passing in.
    def keyframe(path):
        return ImageOps.exif_transpose(Image.open(path)).convert('RGB') if path else None

    if verbose:
        print(f'\n  Denoising {steps} steps at {width}x{height}, {num_frames} frames ...')
    t0 = time.time()

    outputs = ["videos", "audio", "sampling_rate"]
    state = pipe(
        prompt=prompt,
        image=keyframe(image_path),
        last_image=keyframe(last_image_path),
        height=height,
        width=width,
        num_frames=num_frames,
        num_inference_steps=int(steps),
        generator=torch.Generator('cpu').manual_seed(int(seed)),
        output=outputs,
    )
    generate_seconds = time.time() - t0
    if verbose:
        print(f'  Denoise+decode done in {generate_seconds:.0f}s')

    # Free VRAM between denoise and mux
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Phase 4: Mux video + audio
    if verbose:
        print(f'\n  Muxing video + audio ...')
    videos = state["videos"]
    audio = state["audio"]
    sampling_rate = state["sampling_rate"]
    frames = videos[0]
    audio_data = audio[0].cpu() if hasattr(audio[0], 'cpu') else audio[0]
    sr = sampling_rate if sampling_rate is not None else 32000

    out_dir = OUT_DIR / f'gen_{int(time.time())}'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = str(out_dir / 'output.mp4')
    encode_video(frames, fps=FPS, output_path=out_path,
                 audio=audio_data, audio_sample_rate=sr)
    if verbose:
        sz = os.path.getsize(out_path) / 1024 / 1024
        print(f'  Output: {out_path} ({sz:.1f} MB)')

    return out_path, {
        'width': width,
        'height': height,
        'num_frames': num_frames,
        'steps': int(steps),
        'seed': int(seed),
        'generate_seconds': generate_seconds,
    }

def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

print('STEP 3 complete — official MiniMax-H3 pipeline (PR #14355) loaded.')
print('Next: run STEP 4 to open the Gradio UI.')


In [ ]:
#@title STEP 4 — Gradio UI (text-to-video + image-to-video with audio)
"""
• Two-column layout: left = controls, right = video output
• Prompt + optional first/last frame images
• Canvas selector (aspect ratios)
• Duration slider (5-14 seconds)
• Steps slider (10-40)
• Seed (0 or negative = random)
• Prompt rewrite toggle
"""
import os, sys, time, pathlib, traceback, random
import torch
import gradio as gr

CSS = """
#col-container   { margin: 0 auto; max-width: 1400px; }
#main-title h1   { font-size: 2.4em !important; }
"""

with gr.Blocks(css=CSS, delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **MiniMax-H3 — Video + Audio Generation**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Generate video with synchronized audio from a text prompt '
        '(optionally with first/last frame images). '
        'Powered by [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) (33B, INT8 quantized).'
    )

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            prompt = gr.Textbox(
                label='Prompt',
                lines=3,
                value='A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot',
                info='Describe the scene. Audio (ambience, foley, speech) is generated automatically.',
            )
            with gr.Row():
                image = gr.Image(label='First frame (optional)', type='filepath', height=200)
                last_image = gr.Image(label='Last frame (optional)', type='filepath', height=200)
            btn_generate = gr.Button('Generate video', variant='primary')
            with gr.Accordion('Advanced options', open=False):
                canvas = gr.Dropdown(
                    label='Canvas (aspect ratio)',
                    choices=list(CANVASES.keys()),
                    value=DEFAULT_CANVAS,
                    info='Resolution and aspect ratio. "fast" variants are smaller and quicker.',
                )
                duration = gr.Slider(
                    5, MAX_UI_DURATION, value=5, step=1,
                    label='Duration (seconds)',
                    info='Snapped to the nearest valid frame count (17*n+5 at 24 fps). MiniMax-H3 generates 5-15s.',
                )
                steps = gr.Slider(
                    10, 40, value=28, step=1,
                    label='Inference steps',
                    info='More steps = higher quality but slower. 28 is the default.',
                )
                seed = gr.Number(
                    value=42,
                    label='Seed (0 or negative = random)',
                    info='Different seeds = different videos. Use 0 or negative for a random seed each run.',
                    precision=0,
                )
                rewrite_prompt = gr.Checkbox(
                    value=False,
                    label='Rewrite prompt',
                    info='Hand the prompt to the pipeline unchanged; this is a placeholder for future prompt refinement.',
                )
            status_box = gr.Textbox(
                label='Status', interactive=False, lines=3,
                placeholder='Awaiting generation...',
            )

        with gr.Column(scale=2):
            output_video = gr.Video(label='Video + soundtrack', height=500)
            report_box = gr.Markdown(visible=False)
            with gr.Accordion('Downloads', open=False):
                dl_video = gr.File(label='Download .mp4')

    # --- Event wiring ---
    def generate(prompt_text, img_path, last_img_path, canvas_label,
                  dur, n_steps, s, rewrite, progress=gr.Progress(track_tqdm=True)):
        try:
            if not prompt_text or not prompt_text.strip():
                raise gr.Error('A prompt is required.')
            # Seed: 0 or negative = random
            actual_seed = int(s)
            if actual_seed <= 0:
                actual_seed = random.randint(1, 2**31 - 1)
            progress(0.0, desc='Conditioning ...')
            out_path, report = generate_video(
                prompt=prompt_text,
                image_path=img_path if img_path else None,
                last_image_path=last_img_path if last_img_path else None,
                canvas=canvas_label,
                duration=dur,
                steps=n_steps,
                seed=actual_seed,
                rewrite_prompt=rewrite,
                verbose=True,
            )
            report_md = (
                f'`{report["width"]}x{report["height"]}`, {report["num_frames"]} frames '
                f'({report["num_frames"]/FPS:.1f}s), {report["steps"]} steps · '
                f'denoise {report["generate_seconds"]:.0f}s · seed {report["seed"]}'
            )
            if report.get('refined_prompt'):
                report_md += f'\n\n**Refined prompt:** {report["refined_prompt"]}'
            free_cuda()
            return (
                gr.update(value=out_path),
                gr.update(value=report_md, visible=True),
                gr.update(value=out_path),
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Generation failed: {e}')

    btn_generate.click(
        generate,
        inputs=[prompt, image, last_image, canvas, duration, steps, seed, rewrite_prompt],
        outputs=[output_video, report_box, dl_video],
    )

    def _welcome():
        return (
            'Enter a prompt, optionally upload first/last frame images, '
            'then click "Generate video". Use seed 0 for random. '
            'First run downloads weights (~134 GB) then loads + quantizes the model '
            '(~5-10 min after STEP 2).'
        )
    demo.load(_welcome, inputs=None, outputs=[status_box])

# --- Queue + launch ────────────────────────────────────────────────────
demo.queue(default_concurrency_limit=2, max_size=4)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(share=False, server_name='0.0.0.0', server_port=7860, show_error=True, height=1100)


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""Standard AEI-suite keep-alive cell."""
import os, sys, time, pathlib
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.')
print('='*72)

try:
    summary = {
        'cache_root'    : str(drive_root),
        'ckpt_dir'      : str(CKPT_DIR),
        'out_dir'       : str(OUT_DIR),
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'diffusers'     : diffusers.__version__,
        'transformers'  : transformers.__version__,
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).')


In [ ]:
#@title STEP 6 — Quick test (single video generation)
"""Stand-alone test. Generates one video with default parameters."""
import os, sys, time, pathlib, random
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — single-video quick test')
print('='*72)

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot'  #@param {type:"string"}
IMAGE_PATH = ''  #@param {type:"string"}
LAST_IMAGE_PATH = ''  #@param {type:"string"}
CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '640x1152 · 9:16', '544x544 · 1:1 fast', '768x768 · 1:1 full']
DURATION = 5  #@param {type:"slider", min:5, max:14, step:1}
STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
SEED = 42  #@param {type:"integer"}
REWRITE = False  #@param {type:"boolean"}

# Seed: 0 or negative = random
if SEED <= 0:
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

img_path = IMAGE_PATH.strip() if IMAGE_PATH.strip() else None
last_img_path = LAST_IMAGE_PATH.strip() if LAST_IMAGE_PATH.strip() else None

print(f'  Prompt      : {PROMPT[:60]}...')
print(f'  First frame : {img_path or "(none)"}')
print(f'  Last frame  : {last_img_path or "(none)"}')
print(f'  Canvas      : {CANVAS}')
print(f'  Duration    : {DURATION}s')
print(f'  Steps       : {STEPS}')
print(f'  Seed        : {SEED}')
print()

t0 = time.time()
out_path, report = generate_video(
    prompt=PROMPT,
    image_path=img_path,
    last_image_path=last_img_path,
    canvas=CANVAS,
    duration=DURATION,
    steps=STEPS,
    seed=SEED,
    rewrite_prompt=REWRITE,
    verbose=True,
)
total_time = time.time() - t0

print()
print('='*72)
print(f'Video generated in {total_time:.0f}s')
print(f'  Resolution : {report["width"]}x{report["height"]}')
print(f'  Frames     : {report["num_frames"]} ({report["num_frames"]/FPS:.1f}s)')
print(f'  Denoise    : {report["generate_seconds"]:.0f}s')
print(f'  Seed       : {report["seed"]}')
print(f'  Output     : {out_path}')
sz = os.path.getsize(out_path) / 1024 / 1024
print(f'  Size       : {sz:.1f} MB')
print('='*72)

display(FileLink(out_path, result_html_prefix='Download video: '))
free_cuda()
print('\nSTEP 6 complete. Open the Gradio UI (STEP 4) for the full experience.')


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list
"""
Advanced batch processor. Reads a JSON file containing a list of scenes,
each with its own prompt, optional keyframe images, canvas, duration,
steps, and seed. The model is loaded once and reused.

JSON format (a list of objects):
```json
[
  {
    "prompt": "A red fox trotting through a snowy pine forest at dawn",
    "image": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_image": "/content/drive/MyDrive/keyframes/fox_end.png",
    "canvas": "960x544 · 16:9 fast",
    "duration": 6,
    "steps": 28,
    "seed": 42
  },
  {
    "prompt": "A busy night market, neon signs reflecting in puddles",
    "canvas": "544x960 · 9:16 fast",
    "duration": 5,
    "seed": 0
  }
]
```

Fields (all optional except `prompt`):
  - prompt:      (required) text description of the scene
  - image:       (optional) path to first frame image
  - last_image:  (optional) path to last frame image
  - canvas:      (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:    (optional, default 5) seconds (5-14)
  - steps:       (optional, default 28) inference steps (10-40)
  - seed:        (optional, default 0 = random) 0 or negative = random
  - rewrite:     (optional, default false) rewrite prompt via conditioner

A progress log is written to batch_log.jsonl for resume after disconnect.
"""
import os, sys, time, json, pathlib, traceback, random
import torch
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — Batch generation (JSON scene list)')
print('='*72)

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-H3/batch_scenes.json'  #@param {type:"string"}
DEFAULT_DURATION = 5  #@param {type:"slider", min:5, max:14, step:1}
DEFAULT_STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
DEFAULT_CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '640x1152 · 9:16', '768x1344 · 9:16 full', '544x544 · 1:1 fast', '768x768 · 1:1 full', '768x576 · 4:3 fast', '1024x768 · 4:3 full', '576x768 · 3:4 fast', '768x1024 · 3:4 full', '1152x512 · 21:9 fast', '1536x672 · 21:9 full']
SKIP_EXISTING = True  #@param {type:"boolean"}
REWRITE_DEFAULT = False  #@param {type:"boolean"}

# --- Load and validate the JSON scene list ────────────────────────────
json_path = pathlib.Path(BATCH_JSON_PATH)
if not json_path.exists():
    # Create a template if the file doesn't exist
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {"prompt": "A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION, "steps": DEFAULT_STEPS, "seed": 42},
        {"prompt": "A busy night market, neon signs reflecting in puddles, sizzling street food",
         "canvas": "544x960 · 9:16 fast", "duration": 5, "steps": 28, "seed": 0},
    ]
    json_path.write_text(json.dumps(template, indent=2, ensure_ascii=False))
    print(f'  [INFO] Template created at {json_path}')
    print(f'  [INFO] Edit it with your scenes, then re-run this cell.')
    raise SystemExit(0)

try:
    scenes = json.loads(json_path.read_text())
except json.JSONDecodeError as e:
    raise SystemExit(f'[ERROR] Invalid JSON in {json_path}: {e}')

if not isinstance(scenes, list):
    raise SystemExit(f'[ERROR] JSON must be a list of scene objects, got {type(scenes).__name__}')

# Validate scenes
valid_scenes = []
for i, scene in enumerate(scenes):
    if not isinstance(scene, dict):
        print(f'  [WARN] Scene {i}: not a dict, skipping')
        continue
    prompt = scene.get('prompt', '').strip()
    if not prompt:
        print(f'  [WARN] Scene {i}: empty prompt, skipping')
        continue
    # Apply defaults
    scene.setdefault('canvas', DEFAULT_CANVAS)
    scene.setdefault('duration', DEFAULT_DURATION)
    scene.setdefault('steps', DEFAULT_STEPS)
    scene.setdefault('seed', 0)
    scene.setdefault('rewrite', REWRITE_DEFAULT)
    # Validate canvas
    if scene['canvas'] not in CANVASES:
        print(f'  [WARN] Scene {i}: canvas not found, using default')
        scene['canvas'] = DEFAULT_CANVAS
    # Validate image paths
    for key in ('image', 'last_image'):
        val = scene.get(key, '')
        if val:
            p = pathlib.Path(val)
            if not p.exists():
                print(f'  [WARN] Scene {i}: {key} "{val}" not found')
                scene[key] = None
            else:
                scene[key] = str(p)
        else:
            scene[key] = None
    valid_scenes.append(scene)

print(f'  JSON file   : {json_path}')
print(f'  Scenes      : {len(valid_scenes)} (of {len(scenes)} parsed)')
print(f'  Defaults    : canvas={DEFAULT_CANVAS}, duration={DEFAULT_DURATION}s, steps={DEFAULT_STEPS}')
print()

# --- Output directory ────────────────────────────────────────────────
output_subdir = OUT_DIR / f'batch_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)
batch_log = output_subdir / 'batch_log.jsonl'

# --- Resume support: check which scenes are already done ─────────────
already_done = set()
if SKIP_EXISTING and batch_log.exists():
    try:
        with open(batch_log) as f:
            for ln in f:
                try:
                    rec = json.loads(ln)
                    if rec.get('status') == 'ok':
                        already_done.add(rec.get('scene_idx', -1))
                except Exception:
                    pass
    except Exception:
        pass
    if already_done:
        print(f'  Resume: skipping {len(already_done)} already-completed scene(s)')

# --- Generate ─────────────────────────────────────────────────────────
results = []
batch_start = time.time()

with open(batch_log, 'a', buffering=1) as log_f:
    for i, scene in enumerate(valid_scenes):
        scene_idx = i + 1
        prompt = scene['prompt']
        img = scene.get('image')
        last_img = scene.get('last_image')
        canvas = scene['canvas']
        duration = scene['duration']
        steps = scene['steps']
        raw_seed = scene['seed']
        rewrite = scene.get('rewrite', False)

        # Seed: 0 or negative = random
        if raw_seed <= 0:
            seed = random.randint(1, 2**31 - 1)
        else:
            seed = int(raw_seed)

        # Skip if already done
        if scene_idx in already_done:
            print(f'  [{scene_idx:03d}/{len(valid_scenes)}] SKIP (already done)')
            results.append(('skipped', prompt, None))
            continue

        # Print scene info
        print(f'  [{scene_idx:03d}/{len(valid_scenes)}] seed={seed} {prompt[:60]}...')
        if img:
            print(f'         first_frame: {pathlib.Path(img).name}')
        if last_img:
            print(f'         last_frame:  {pathlib.Path(last_img).name}')
        print(f'         canvas: {canvas}, duration: {duration}s, steps: {steps}')

        t0 = time.time()
        try:
            out_path, report = generate_video(
                prompt=prompt,
                image_path=img,
                last_image_path=last_img,
                canvas=canvas,
                duration=duration,
                steps=steps,
                seed=seed,
                rewrite_prompt=rewrite,
                verbose=False,
            )
            elapsed = time.time() - t0
            sz = os.path.getsize(out_path) / 1024 / 1024
            print(f'    -> OK ({elapsed:.0f}s, {sz:.1f} MB, seed={report["seed"]})')
            results.append(('ok', prompt, out_path))

            # Copy to a named file for easy identification
            named_path = output_subdir / f'scene_{scene_idx:03d}.mp4'
            import shutil
            shutil.copy2(out_path, named_path)

            log_f.write(json.dumps({
                'scene_idx': scene_idx,
                'prompt': prompt,
                'status': 'ok',
                'elapsed_s': elapsed,
                'video': str(named_path),
                'original_video': out_path,
                'seed': report['seed'],
                'width': report['width'],
                'height': report['height'],
                'frames': report['num_frames'],
                'size_mb': sz,
                'canvas': canvas,
                'duration': duration,
                'steps': steps,
                'image': img,
                'last_image': last_img,
            }) + '\n')
        except Exception as e:
            elapsed = time.time() - t0
            print(f'    -> FAIL ({elapsed:.0f}s): {e}')
            traceback.print_exc(limit=2)
            results.append(('error', prompt, str(e)))
            log_f.write(json.dumps({
                'scene_idx': scene_idx,
                'prompt': prompt,
                'status': 'error',
                'error': str(e),
                'elapsed_s': elapsed,
                'seed': seed,
            }) + '\n')
        free_cuda()

total_elapsed = time.time() - batch_start
n_ok = sum(1 for r in results if r[0] == 'ok')
n_skip = sum(1 for r in results if r[0] == 'skipped')
n_err = sum(1 for r in results if r[0] == 'error')

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_skip} skipped / {n_err} errors in {total_elapsed:.0f}s')
print(f'  Output: {output_subdir}')
print(f'  Log:   {batch_log}')
print('='*72)

for f in sorted(output_subdir.glob('scene_*.mp4')):
    sz = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:>20s}  {sz:>8.1f} MB')

if n_ok > 0:
    print(f'\n  Tip: zip with `!cd {output_subdir} && zip -r batch.zip .`')
    print(f'  Tip: the JSONL log has all scene metadata for your records')
